# 07 · What the grid's changes are worth: day-ahead prices
Hourly day-ahead settlement prices from ERCOT's public report NP4-180-ER, 2019 to 2025. `HB_BUSAVG` is the average across the hubs' buses. Uri (Feb 10–20 2021) is excluded from the yearly tables and studied separately.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import duckdb, numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
con = duckdb.connect("data/grid.duckdb", read_only=True)
con.execute("SET TimeZone = 'UTC'")
q = lambda sql: con.execute(sql).df()
import plotly.graph_objects as go
from viz import add_caption, save, SERIES, INK
p = pd.read_parquet("data/raw/ercot_dam_prices.parquet")
p["d"] = pd.to_datetime(p.delivery_date)
p = p[~p.repeated_hour]
uri = p.d.between("2021-02-10", "2021-02-20")
P = p[(p.hub == "HB_BUSAVG") & ~uri].pivot_table(index="d", columns="hour_ending", values="price_usd_mwh")
print(P.shape, "days x hours")

(2546, 24) days x hours


## The evening premium
Medians, because a few scarcity days dominate the means. Midday is hours ending 11 am to 3 pm and evening is hours ending 7 to 9 pm. The spread is the top two hours minus the bottom two hours each day, a rough gross margin per MWh for a two-hour battery before losses.

In [2]:
def table(frame):
    srt = np.sort(frame.dropna().values, axis=1)
    return pd.DataFrame({"midday": frame.loc[:, 11:15].mean(axis=1).groupby(frame.index.year).median(),
                         "evening": frame.loc[:, 19:21].mean(axis=1).groupby(frame.index.year).median(),
                         "daily spread": pd.Series(srt[:, -2:].mean(axis=1) - srt[:, :2].mean(axis=1), index=frame.dropna().index).groupby(frame.dropna().index.year).median()})
annual, spring = table(P), table(P[P.index.month.isin([3, 4, 5])])
annual["evening premium"], spring["evening premium"] = annual.evening - annual.midday, spring.evening - spring.midday
annual.round(1).to_csv("data/clean/price_premium_annual.csv"); spring.round(1).to_csv("data/clean/price_premium_spring.csv")
display(annual.round(1)); spring.round(1)

,midday,evening,daily spread,evening premium
d,,,,
2019,25.3,27.4,28.4,2.1
2020,21.2,23.3,24.9,2.2
2021,31.7,38.8,40.0,7.1
2022,47.6,79.2,66.5,31.7
2023,23.6,41.9,43.7,18.3
2024,17.8,39.8,36.2,22.0
2025,23.0,52.5,46.4,29.5


,midday,evening,daily spread,evening premium
d,,,,
2019,25.1,27.6,24.7,2.5
2020,20.8,23.1,25.1,2.3
2021,23.9,30.1,27.9,6.2
2022,45.7,79.3,84.8,33.6
2023,20.6,39.9,37.4,19.2
2024,13.0,42.1,46.8,29.1
2025,20.3,56.0,50.6,35.7


In [3]:
fig = go.Figure()
for i, (col, name) in enumerate([("evening", "Evening, 7 to 9 pm"), ("midday", "Midday, 11 am to 3 pm")]):
    fig.add_scatter(x=spring.index, y=spring[col], mode="lines+markers", name=name, line=dict(color=SERIES[i], width=2.5), marker=dict(size=8),
                    hovertemplate=name + " · %{x}: $%{y:.0f}/MWh<extra></extra>")
    fig.add_annotation(x=2025, y=spring[col].iloc[-1], text=f"{name.split(',')[0]} ${spring[col].iloc[-1]:.0f}", showarrow=False, xanchor="left", xshift=8, font=dict(color=INK))
fig.update_layout(title="Spring day-ahead prices: evenings have pulled away from midday", xaxis_title="Year", yaxis_title="Median price ($/MWh)",
                  xaxis=dict(tickvals=list(range(2019, 2026)), range=[2018.7, 2026.1]), yaxis=dict(rangemode="tozero"), margin=dict(r=130))
add_caption(fig, f"The spring evening premium rose from {spring['evening premium'][2019]:.1f} to {spring['evening premium'][2025]:.1f} dollars per MWh. 2022 reflects high gas prices. HB_BUSAVG, March to May medians.")
save(fig, "evening_premium")

## Winter Storm Uri at day-ahead prices
Day-ahead price times served demand. This is a measure of market value, not what customers paid: hedges, real-time settlement, and later securitization all change the final bill.

In [4]:
b = p[p.hub == "HB_BUSAVG"]
d = q("SELECT local_ts, local_hour, demand_mwh FROM model_hourly")
d["local_ts"] = pd.to_datetime(d.local_ts)
d["d"] = (d.local_ts - pd.Timedelta(hours=1)).dt.normalize(); d["hour_ending"] = d.local_hour.replace(0, 24)
m = d.merge(b, on=["d", "hour_ending"]); m["value"] = m.demand_mwh * m.price_usd_mwh
yearly = (m.groupby(m.d.dt.year).value.sum() / 1e9).round(1).rename("$bn")
u = m[m.d.between("2021-02-14", "2021-02-19")]
before = m[m.d.between("2021-01-20", "2021-02-09")]
print(f"Feb 14-19 2021: ${u.value.sum() / 1e9:.1f}bn on {u.demand_mwh.sum() / 1000:,.0f} GWh, average ${u.value.sum() / u.demand_mwh.sum():,.0f}/MWh, max ${u.price_usd_mwh.max():,.0f}")
print(f"Jan 20-Feb 9 2021 average: ${before.value.sum() / before.demand_mwh.sum():.2f}/MWh")
yearly

Feb 14-19 2021: $47.4bn on 7,269 GWh, average $6,522/MWh, max $8,996
Jan 20-Feb 9 2021 average: $21.64/MWh


d
2019    18.2
2020     9.2
2021    63.7
2022    31.7
2023    32.7
2024    14.1
2025    16.9
Name: $bn, dtype: float64

## West Texas: from cheap power to a premium
West hub minus North hub. Through 2022 West was cheaper, as wind crowded export lines. If local load outgrows the wires, the sign flips.

In [5]:
h = p[~uri].pivot_table(index=["d", "hour_ending"], columns="hub", values="price_usd_mwh")
gap = h.HB_WEST - h.HB_NORTH
yrs = gap.index.get_level_values(0).year
west = pd.DataFrame({"West minus North ($/MWh)": gap.groupby(yrs).mean(), "% of hours West is $5+ above North": 100 * (gap > 5).groupby(yrs).mean()}).round(1)
west.to_csv("data/clean/west_hub_gap.csv")
west

,West minus North ($/MWh),% of hours West is $5+ above North
d,,
2019,-1.4,0.2
2020,-2.0,0.9
2021,-4.1,0.2
2022,-4.8,4.5
2023,1.0,19.5
2024,1.9,22.6
2025,0.8,28.2
